[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/p1_muscle_forces.ipynb)

# Subject P1 walking, muscle forces: which muscles carry each joint torque

Inverse dynamics gives me one net torque per joint. It doesn't say who makes it.
So here I hand P1's walking (trial 13, the committed sample) to OpenSim, get
muscle forces two ways (static optimization and dynamic optimization), and
line them up against the torques, all on the same clock as the animation.

What this notebook delivers:

1. **The walking animation**, the whole 12-segment body, looping. Open
   Controls &rarr; Animations to pause and scrub.
2. **The 3&times;3 sagittal scope** underneath (GRF on top, then angle,
   angular velocity and torque for the right ankle, knee and hip), scrolling
   with the animation.
3. **A muscle grid on the same clock**: ankle / knee / hip columns, with
   static-optimization forces, dynamic-optimization forces, and the summed
   muscle moment over the inverse-dynamics torque.
4. **Checkboxes per muscle**, so you can untick muscles and watch how much of
   the torque is left (biarticular muscles show up in every column they cross).
5. **Two optional rows under the moment row**, off by default with a tick box each:
   normalized fibre length l<sub>M</sub>/l<sub>opt</sub> and fibre velocity
   v<sub>M</sub>/(v<sub>max</sub> l<sub>opt</sub>), v<sub>max</sub> = 10 l<sub>opt</sub>/s. One
   trace per muscle, because SO and dynamic optimization see the same fibres (2.4e-4 RMS
   apart in length, 7.3e-4 in velocity).
6. **The muscle model and the two optimizations**: the DeGrooteFregly2016 equations as
   OpenSim runs them (f<sub>L</sub> widened &times;1.5, f<sub>V</sub>, damping 0.1, passive and
   tendon curves shown but off), activation dynamics with &tau; = 10/40 ms and b = 0.1 (so
   90% on in 41 ms and 90% off in 37 ms), where each muscle works on f<sub>L</sub> and
   f<sub>V</sub> (hamstrings at 1.29-1.32 l<sub>opt</sub> when loaded, vasti at 0.88, none of the 16
   biggest shortening faster than &minus;0.48 v<sub>max</sub> while loaded), and SO vs MocoInverse written out exactly.
7. **Knee contact forces** (right tibiofemoral, static vs dynamic optimization), split into
   medial and lateral with two point contacts 40.3 mm apart. Peak total
   5.83 BW (static) / 6.13 BW (dynamic), medial 4.00 / 3.98 BW, lateral
   2.08 / 2.57 BW, medial share 67% / 62% at the peak. That's about 2.2-2.3&times;
   the 2.61 BW measured in instrumented knees (the medial share is close to the in vivo 65-73%).
8. **A varus sweep**: the right knee pushed laterally 0-50 mm (0-14.2&deg; of varus,
   built into the model too). Peak medial 4.00 &rarr; 5.90 BW, lateral 2.08 &rarr;
   0.86 BW, total 5.83 &rarr; 5.97 BW; frontal (adduction) moment 68.9 &rarr;
   124.7 N m, sagittal (extension) moment 134 &rarr; 148 N m. The lateral condyle
   starts lifting off at d = 20 mm.

The muscle numbers come from `data/opensim/p1_trial13_muscles.npz` (made by the
OpenSim side, not by `boneid`). Everything else is the usual pipeline
(`report.whole_body_run`).

In [1]:
# Colab bootstrap: a no-op when running locally. On Colab it clones the repo (for its
# data/ folder), installs boneid from the clone, and runs from notebooks/ so the
# relative paths below work.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("../src/boneid"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jeremydwong/bonelab_inverse_dynamics"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "bonelab_inverse_dynamics"], check=True)
    os.chdir("bonelab_inverse_dynamics/notebooks")
os.makedirs("../reports", exist_ok=True)

In [2]:
import os
import numpy as np
from pathlib import Path

from IPython.display import HTML, display

from boneid import io_v3d as io
from boneid import report as R

PATH = "../data/p1_trial13_sample.mat"        # trial-13 slot only, index 0
MUSCLES = Path("../" + R.MUSCLE_NPZ)            # data/opensim/p1_trial13_muscles.npz
if not MUSCLES.exists():
    raise FileNotFoundError(f"{MUSCLES} isn't there yet: run the OpenSim step first")

params = R.p1_params()
trial = io.load_v3d_trial(PATH, 0)
mus = R.load_muscles(MUSCLES)
print(f"{len(mus['muscle_names'])} muscles, {len(mus['t'])} samples, "
      f"dynamic-opt window {mus['window'][0]:.2f}-{mus['window'][1]:.2f} s")
meta = mus["meta"]
print(f"model: {meta['model'].split(' (')[0]}; IK marker RMS {1000 * meta['ik']['rms_mean_m']:.1f} mm mean; "
      f"static optimization {meta['static_optimization']['runtime_s']:.0f} s, "
      f"dynamic optimization (MocoInverse, IPOPT) {meta['dynamic_optimization'].get('runtime_s', float('nan')):.0f} s")

# OpenSim's sign convention is ours times [-1, +1, -1] (ankle, knee, hip; see the
# sidecar's conventions), so flip those columns to match the torque row of the scope.
SIGN = np.array([-1.0, 1.0, -1.0])
mus["moment_arm"] = mus["moment_arm"] * SIGN
for k in ("so_moment", "dyn_moment", "id_moment_opensim", "id_moment_boneid"):
    mus[k] = mus[k] * SIGN
meta["moment_convention"] = "boneid"

40 muscles, 637 samples, dynamic-opt window 0.10-5.20 s
model: RajagopalLaiUhlrich2023; IK marker RMS 14.2 mm mean; static optimization 58 s, dynamic optimization (MocoInverse, IPOPT) 1087 s


## The pipeline and the animation

Same calls as the P1 report: both legs, pelvis, torso and arms, then the meshcat
clip (30 fps, decimated 4 from 120 Hz).

In [3]:
run = R.whole_body_run(trial, params)
chain, skel_u, kin_u, whole, audit = run
viewer_html = R.p1_viewer_html(trial, chain, skel_u, kin_u, params)
n_key, fps, period = R.viewer_period(chain[1])
print(f"clip: {n_key} keyframes at {fps:.0f} fps, period {period:.2f} s")

clip: 160 keyframes at 30 fps, period 5.30 s


## The page

`report.muscle_viewer_html` puts the viewer, the 3&times;3 scope and the muscle
grid together. I flipped row (c) into our sign convention (above), so it reads
the same way as the torque row of the scope.

In [4]:
html = R.muscle_viewer_html(trial, mus, params, viewer_html=viewer_html,
                            run=run)
Path("../reports").mkdir(exist_ok=True)
Path("../reports/p1_muscle_forces_page.html").write_text(html)
display(HTML(html))

## The muscle model and the two optimizations: the equations OpenSim actually ran, and why static and dynamic land close

Both solves use the same muscle: OpenSim's `DeGrooteFregly2016Muscle` (De Groote, Kinney,
Rao & Fregly 2016, *Ann Biomed Eng* 44:2922, doi 10.1007/s10439-016-1591-9, 308 citations on
Semantic Scholar). I swapped the Rajagopal model's Millard muscles for it with
`ModOpReplaceMusclesWithDeGrooteFregly2016`, then switched off tendon compliance and passive fibre
force and widened the active force-length curve by 1.5 (the Moco walking-example settings).
Below are the equations as OpenSim 4.6 implements them (from `DeGrooteFregly2016Muscle.h`;
the constants differ a little from the paper's supplement, and the header says why). I
checked every curve against OpenSim's own outputs (numbers in the next cell).

**Notation.** $\tilde l^M = l^M / l^M_{opt}$ (normalized fibre length),
$\tilde v^M = v^M / (v_{max}\, l^M_{opt})$ with $v_{max} = 10$ optimal lengths per second for every
muscle here (the model's `max_contraction_velocity`; positive = lengthening),
$\tilde l^T = l^T / l^T_s$ (tendon over slack length), $\alpha$ = pennation angle.

**Force along the tendon** (what MocoInverse applies):

$$F^T = F^M_{iso}\,\Big(a\, f_L(\tilde l^M)\, f_V(\tilde v^M) \;+\; f_{PE}(\tilde l^M) \;+\; \beta\,\tilde v^M\Big)\cos\alpha$$

with $f_{PE} \equiv 0$ here (passive force ignored) and $\beta = 0.1$, a linear fibre damping
that the Rajagopal model's muscles carry and the replacement kept. (I didn't know about the
damping term until I rebuilt the forces: MocoInverse's tendon force reconstructs from this formula to
$3\times10^{-11}$ N, and leaving $\beta$ out misses by up to 343 N.)

**Rigid-tendon geometry.** Fibre width is constant, $l^M \sin\alpha = l^M_{opt}\sin\alpha_{opt}$,
and the tendon never stretches, so

$$l^{MT}(q) = l^T_s + l^M\cos\alpha \quad\Rightarrow\quad l^M = \sqrt{(l^{MT} - l^T_s)^2 + (l^M_{opt}\sin\alpha_{opt})^2}, \qquad v^M = \frac{d l^M}{dt}.$$

So fibre length and velocity are pure functions of the joint angles and their rates. That's
the whole reason the two optimizations see the same fibres.

**Active force-length** (a sum of three Gaussian-like bumps, evaluated at a stretched length):

$$x = \frac{\tilde l^M - 1}{w} + 1, \quad w = 1.5, \qquad f_L = \sum_{i=1}^{3} b_{1i}\,\exp\!\left(-\frac{1}{2}\,\frac{(x - b_{2i})^2}{(b_{3i} + b_{4i}\,x)^2}\right)$$

| $i$ | $b_{1i}$ | $b_{2i}$ | $b_{3i}$ | $b_{4i}$ |
|---|---|---|---|---|
| 1 | 0.8150671134 | 1.0550334290 | 0.1623845736 | 0.0633034485 |
| 2 | 0.4330049844 | 0.7167754134 | &minus;0.0299471170 | 0.2003568473 |
| 3 | 0.1 | 1.0 | 0.3535533906 ($=\tfrac12\sqrt{\tfrac12}$) | 0 |

The $\times1.5$ width is the `active_force_width_scale` property (set on all 80 muscles by
`ModOpScaleActiveFiberForceCurveWidthDGF(1.5)`). It only rescales the *input*: the curve keeps
its peak $f_L(1) = 1$, but a fibre has to move 1.5 times as far from $l_{opt}$ to lose the same
fraction of force. ($b_{11}$ is OpenSim's tweak so that $f_L(1) = 1$ exactly.)

**Force-velocity** (an inverse-sinh, so it can be inverted in closed form):

$$f_V = d_1 \ln\!\Big[(d_2\tilde v^M + d_3) + \sqrt{(d_2\tilde v^M + d_3)^2 + 1}\Big] + d_4$$

with $d_1 = -0.3211346128$, $d_2 = -8.149$, $d_3 = -0.374$, $d_4 = 0.8825327733$. OpenSim re-solved
$d_1, d_4$ so the curve goes exactly through $(-1, 0)$ (full-speed shortening, no force) and
$(0, 1)$. It tops out near 1.8 for fast lengthening.

**Passive force-length** (shown for completeness, **ignored** in both solves):

$$f_{PE} = \frac{e^{k_{PE}(\tilde l^M - 1)/e_0} - e^{k_{PE}(0.2 - 1)/e_0}}{e^{k_{PE}} - e^{k_{PE}(0.2 - 1)/e_0}}, \qquad k_{PE} = 4,$$

with $e_0$ = `passive_fiber_strain_at_one_norm_force`, 0.50&ndash;0.90 across these muscles
(median 0.70, carried over from the Rajagopal/Uhlrich model).

**Tendon force-length** (shown, **not used**: the tendon is rigid):

$$f_T = c_1\, e^{k_T(\tilde l^T - c_2)} - c_3, \qquad c_1 = 0.2,\; c_2 = 1,\; c_3 = 0.2,\; k_T = \frac{\ln\big((1 + c_3)/c_1\big)}{1 + \varepsilon_0 - c_2} = \frac{\ln 6}{0.049} = 36.6$$

**Activation dynamics** (only the dynamic optimization has them):

$$\frac{da}{dt} = \left[\frac{f + \tfrac12}{\tau_a\,(0.5 + 1.5a)} + \frac{(0.5 + 1.5a)\,(\tfrac12 - f)}{\tau_d}\right](e - a), \qquad f = \tfrac12\tanh\!\big(b\,(e - a)\big)$$

The `tanh` is a smooth switch between the activation time constant (when $e > a$) and the
deactivation one (when $e < a$). Two things surprised me here, and I think they're worth
saying plainly:

- The time constants are **$\tau_a = 10$ ms and $\tau_d = 40$ ms**, not DGF's defaults of
  15/60 ms. The muscle replacement copies them from the Rajagopal model's muscles. (The sidecar
  JSON used to say 15/60; I fixed it.)
- The smoothing is $b = 0.1$, the default, which matches the original paper. With $|e - a| \le 1$
  that makes $\tanh(b(e-a))$ almost flat, so the switch never really switches: both time
  constants get about half weight all the time. The step responses in the figure show it: 90%
  activation in 41 ms and 90% deactivation in 37 ms, so deactivation is as fast as activation.
  With $b = 10$ (what the OpenSim docs now recommend) it'd be 33 ms on and 82 ms off.

I checked this ODE against OpenSim's own state derivative at five $(a, e)$ pairs, and it agrees
to all printed digits.


In [5]:
import json

FK = mus["meta"]["fiber_kinematics"]
DO = mus["meta"]["dynamic_optimization"]
SOP = mus["meta"]["static_optimization"]
K = FK["model_constants"]
print("model constants (all 40 right-leg muscles share them):")
for k, v in K.items():
    print(f"  {k:34s} {v}")
print(f"  v_max (optimal lengths/s)          {FK['v_max_lopt_per_s']}")
print(f"\nMocoInverse: {DO['transcription_scheme']}, {DO['num_mesh_intervals']} mesh intervals "
      f"({1000 * DO['mesh_interval_s']:.0f} ms), {DO['multibody_dynamics_mode']} multibody dynamics, "
      f"{DO['optim_solver']} with {DO['optim_hessian_approximation']} Hessian, tolerances "
      f"{DO['optim_convergence_tolerance']:g}/{DO['optim_constraint_tolerance']:g}; "
      f"{DO['num_iterations']} iterations, {DO['runtime_s']:.0f} s, objective {DO['objective']:.1f}, {DO['status']}")
print(f"Static optimization: activation exponent {SOP['activation_exponent']}, use_muscle_physiology "
      f"{SOP['use_muscle_physiology']}, convergence {SOP['convergence_criterion']:g}, {SOP['runtime_s']:.0f} s")
print("\nSO vs MocoInverse fibre kinematics (SO's AnalyzeTool grid interpolated onto the Moco mesh):")
for k, a in FK["so_vs_dyn_agreement"].items():
    print(f"  {k:19s} max {a['max_abs_diff']:.1e}  rms {a['rms_diff']:.1e}  "
          f"p99 {a['p99_abs_diff']:.1e}  (max: {a['argmax_t_s']:.2f} s, {a['argmax_muscle']})")
print(f"  rigid tendon: |l_T - l_slack| max {FK['tendon_minus_slack_max_abs_m']:.1e} m; "
      f"l_MT - l_T - l_M cos(alpha) max {FK['geometry_identity_max_abs_m']:.1e} m")
ac = FK["activation_ode_check"]
print(f"activation ODE vs OpenSim's own da/dt ({ac['muscle']}, {ac['pairs']} (a, e) pairs): "
      f"max diff {ac['max_abs_diff_per_s']:.1e} 1/s")
print("\nreconstructing each solver's reported force from the curves (max |error| over all muscles, N):")
for k, v in FK["force_reconstruction"].items():
    if k != "units":
        print(f"  {k:46s} {v:.3g}")

def show(fig):
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

model constants (all 40 right-leg muscles share them):
  activation_time_constant_s         [0.01]
  deactivation_time_constant_s       [0.04]
  activation_dynamics_smoothing_b    [0.1]
  active_force_width_scale           [1.5]
  fiber_damping                      [0.1]
  ignore_passive_fiber_force         [True]
  ignore_tendon_compliance           [True]
  ignore_activation_dynamics         [False]
  tendon_strain_at_one_norm_force    [0.049]
  v_max (optimal lengths/s)          [10.0]

MocoInverse: hermite-simpson, 256 mesh intervals (20 ms), implicit multibody dynamics, ipopt with limited-memory Hessian, tolerances 0.001/0.001; 335 iterations, 1087 s, objective 153.0, Solve_Succeeded
Static optimization: activation exponent 2, use_muscle_physiology True, convergence 0.0001, 58 s

SO vs MocoInverse fibre kinematics (SO's AnalyzeTool grid interpolated onto the Moco mesh):
  fiber_length_norm   max 1.9e-02  rms 2.4e-04  p99 6.3e-04  (max: 0.43 s, glmax1_r)
  fiber_velocity_norm max 9

The curves themselves. Blue is what the solves used, grey is what's in the model but switched
off (or what the published width would have been). Bottom row: the excitation-to-activation
step response, from 0.01 to 1 and back, solid for our $b = 0.1$ and dashed for $b = 10$, and
the switch weight $0.5 + f$ that decides how much of the activation vs deactivation constant you
get.

In [6]:
B = [(0.8150671134243542, 1.055033428970575, 0.162384573599574, 0.063303448465465),
     (0.433004984392647, 0.716775413397760, -0.029947116970696, 0.200356847296188),
     (0.1, 1.0, 0.353553390593274, 0.0)]
D1, D2, D3, D4 = -0.3211346127989808, -8.149, -0.374, 0.8825327733249912
KPE, C1, C2, C3 = 4.0, 0.2, 1.0, 0.2

def f_L(l, w=1.5):
    x = (l - 1.0) / w + 1.0
    return sum(b1 * np.exp(-0.5 * (x - b2) ** 2 / (b3 + b4 * x) ** 2) for b1, b2, b3, b4 in B)

def f_V(v):
    u = D2 * v + D3
    return D1 * np.log(u + np.sqrt(u * u + 1.0)) + D4

def f_PE(l, e0=0.6):
    off = np.exp(KPE * (0.2 - 1.0) / e0)
    return (np.exp(KPE * (l - 1.0) / e0) - off) / (np.exp(KPE) - off)

def f_T(lt, eps0=0.049):
    kT = np.log((1 + C3) / C1) / (1 + eps0 - C2)
    return C1 * np.exp(kT * (lt - C2)) - C3

def act_rate(a, e, b, ta, td):
    f = 0.5 * np.tanh(b * (e - a))
    z = 0.5 + 1.5 * a
    return ((f + 0.5) / (ta * z) + (0.5 - f) * z / td) * (e - a)

def step(a0, e, b, ta, td, T=0.3, dt=1e-5):
    a = np.empty(int(T / dt) + 1); a[0] = a0
    for i in range(len(a) - 1):          # RK4 on the scalar ODE
        k1 = act_rate(a[i], e, b, ta, td)
        k2 = act_rate(a[i] + 0.5 * dt * k1, e, b, ta, td)
        k3 = act_rate(a[i] + 0.5 * dt * k2, e, b, ta, td)
        k4 = act_rate(a[i] + dt * k3, e, b, ta, td)
        a[i + 1] = a[i] + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return np.arange(len(a)) * dt, a

def t90(t, a, a0, e):                    # time to 90% of the step
    return t[np.argmax(np.abs(a - a0) >= 0.9 * abs(e - a0))]

TA, TD = K["activation_time_constant_s"][0], K["deactivation_time_constant_s"][0]
BS = K["activation_dynamics_smoothing_b"][0]
WIDTH = K["active_force_width_scale"][0]
steps = {}
for b in (BS, 10.0):
    for a0, e in ((0.01, 1.0), (1.0, 0.01)):
        tt, aa = step(a0, e, b, TA, TD)
        steps[(b, a0)] = (tt, aa, t90(tt, aa, a0, e))
for (b, a0), (_, _, t9) in steps.items():
    print(f"b = {b:g}: {'activation 0.01->1' if a0 < 0.5 else 'deactivation 1->0.01'} reaches 90% in {1000 * t9:.1f} ms")

C0, C1c, INK = R.JOINT_COLORS[0], R.JOINT_COLORS[1], "#1c1b18"
fig, axes = R.new_grid(2, 3, height=6.2, width=12.5)
l = np.linspace(0.2, 1.8, 400)
ax = axes[0, 0]
ax.plot(l, f_L(l, 1.0), color=R.FAINT, lw=1.6, ls="--", label="published width (x1)")
ax.plot(l, f_L(l, WIDTH), color=C0, lw=2.0, label=f"what we ran (x{WIDTH:g})")
ax.axvline(1, color=R.FAINT, lw=0.8)
R.style_axes(ax, "fibre length  l$_M$ / l$_{opt}$", "f$_L$")
ax.set_title("active force-length", fontsize=10, color=R.NEUTRAL)
ax.legend(frameon=False, fontsize=8)
ax = axes[0, 1]
v = np.linspace(-1, 1, 400)
ax.plot(v, f_V(v), color=C0, lw=2.0)
ax.axvline(0, color=R.FAINT, lw=0.8)
ax.plot([-1, 0], [0, 1], "o", color=C0, ms=4)
R.style_axes(ax, "fibre velocity  v$_M$ / (v$_{max}$ l$_{opt}$)   (+ = lengthening)", "f$_V$")
ax.set_title("force-velocity", fontsize=10, color=R.NEUTRAL)
ax = axes[0, 2]
e0s = mus["passive_strain_e0"]
for e0, ls in ((e0s.min(), ":"), (np.median(e0s), "-"), (e0s.max(), "--")):
    ax.plot(l, f_PE(l, e0), color=R.FAINT, lw=1.6, ls=ls, label=f"e$_0$ = {e0:.2f}")
ax.text(0.25, 1.6, "IGNORED in both solves\n(ignore_passive_fiber_force)", fontsize=8, color=R.NEUTRAL)
ax.set_ylim(-0.05, 2.2)
R.style_axes(ax, "fibre length  l$_M$ / l$_{opt}$", "f$_{PE}$")
ax.set_title("passive force-length", fontsize=10, color=R.NEUTRAL)
ax.legend(frameon=False, fontsize=8, loc="center left")
ax = axes[1, 0]
lt = np.linspace(0.99, 1.07, 300)
ax.plot(lt, f_T(lt), color=R.FAINT, lw=1.8)
ax.axvline(1.0, color=C0, lw=2.0)
ax.text(1.003, 1.4, "rigid tendon:\nl$_T$ = l$_s$ always", fontsize=8, color=C0)
ax.set_ylim(-0.25, 2.0)
R.style_axes(ax, "tendon length  l$_T$ / l$_s$", "f$_T$")
ax.set_title("tendon force-length (not used)", fontsize=10, color=R.NEUTRAL)
ax = axes[1, 1]
for (b, a0), (tt, aa, t9) in steps.items():
    ls = "-" if b == BS else "--"
    c = C0 if a0 < 0.5 else C1c
    lab = (f"{'on ' if a0 < 0.5 else 'off'} b = {b:g}: 90% in {1000 * t9:.0f} ms")
    ax.plot(1000 * tt, aa, color=c, lw=2.0 if b == BS else 1.2, ls=ls, label=lab)
R.style_axes(ax, "time after the excitation step (ms)", "activation a")
ax.set_xlim(0, 200)
ax.set_title(f"activation dynamics (dyn-opt only), tau {1000 * TA:.0f}/{1000 * TD:.0f} ms", fontsize=10, color=R.NEUTRAL)
ax.legend(frameon=False, fontsize=7.5, loc="center right")
ax = axes[1, 2]
x = np.linspace(-1, 1, 200)
for b, ls in ((BS, "-"), (10.0, "--")):
    ax.plot(x, 0.5 + 0.5 * np.tanh(b * x), color=INK if b == BS else R.FAINT, lw=1.6, ls=ls,
            label=f"b = {b:g}" + (" (ours)" if b == BS else ""))
R.style_axes(ax, "e - a", "weight on the activation time constant")
ax.set_ylim(0, 1)
ax.set_title("the smooth on/off switch, 0.5 + f", fontsize=10, color=R.NEUTRAL)
ax.legend(frameon=False, fontsize=8)
show(fig)

b = 0.1: activation 0.01->1 reaches 90% in 41.1 ms
b = 0.1: deactivation 1->0.01 reaches 90% in 37.4 ms
b = 10: activation 0.01->1 reaches 90% in 32.9 ms
b = 10: deactivation 1->0.01 reaches 90% in 81.5 ms


### Where the muscles actually work on those curves

Here's each of the 16 biggest muscles (by peak dynamic-optimization force) over the whole
dynamic-optimization window. Each row has a grey line from min to max, grey dots for every
sample (a rough density strip), and orange dots for the samples where that muscle is above 30%
of its own peak force. The orange tick is the median while loaded. The grey curve behind is
$f_L$ (left, with our 1.5 width) or $f_V$ (right), on the right-hand axis. Values come from
the MocoInverse solution, and they're the same for static optimization (next cell's numbers).

In [7]:
names = [str(n) for n in mus["muscle_names"]]
L, V, F = mus["fiber_length_norm"], mus["fiber_velocity_norm"], mus["dyn_force"]
ok = np.isfinite(L[:, 0]) & np.isfinite(F[:, 0])
top = np.argsort(-np.nanmax(F[ok], 0))[:16]
fig, axes = R.new_grid(1, 2, height=5.6, width=12.5)
for k, (X, curve, xr, xlab) in enumerate((
        (L, lambda x: f_L(x, WIDTH), (0.2, 1.6), "fibre length  l$_M$ / l$_{opt}$"),
        (V, f_V, (-0.8, 0.5), "fibre velocity  v$_M$ / (v$_{max}$ l$_{opt}$)"))):
    ax = axes[0, k]
    xx = np.linspace(*xr, 300)
    ax2 = ax.twinx()
    ax2.plot(xx, curve(xx), color=R.FAINT, lw=1.4)
    ax2.set_ylim(0, 1.6 if k == 0 else 1.9)
    ax2.set_ylabel("f$_L$" if k == 0 else "f$_V$", color=R.FAINT, fontsize=9)
    ax2.tick_params(colors=R.FAINT, labelsize=8)
    ax2.spines[["top", "left", "bottom"]].set_visible(False)
    ax2.spines["right"].set_color(R.FAINT)
    for r, m in enumerate(top):
        y = len(top) - 1 - r
        x = X[ok, m]
        f = F[ok, m]
        loaded = f > 0.3 * f.max()
        ax.plot([x.min(), x.max()], [y, y], color=R.FAINT, lw=1.0)
        ax.scatter(x, np.full_like(x, y) + np.random.default_rng(m).uniform(-0.18, 0.18, len(x)),
                   s=2, color=R.NEUTRAL, alpha=0.08, lw=0)
        ax.scatter(x[loaded], np.full(loaded.sum(), y), s=6, color=C1c, alpha=0.35, lw=0)
        ax.plot([np.median(x[loaded])] * 2, [y - 0.3, y + 0.3], color=C1c, lw=2.0)
    ax.axvline(1.0 if k == 0 else 0.0, color=R.NEUTRAL, lw=0.8, ls=":")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([names[m].replace("_r", "") for m in top[::-1]], fontsize=8)
    ax.set_xlim(*xr)
    ax.set_ylim(-0.7, len(top) - 0.3)
    R.style_axes(ax, xlab, "")
axes[0, 0].set_title("where on f$_L$ each muscle works (16 biggest, dyn-opt window)", fontsize=10, color=R.NEUTRAL)
axes[0, 1].set_title("where on f$_V$ (- = shortening, + = lengthening)", fontsize=10, color=R.NEUTRAL)
show(fig)
for m in top:
    x, y, f = L[ok, m], V[ok, m], F[ok, m]
    w = f > 0.3 * f.max()
    print(f"{names[m]:11s} peak {f.max():5.0f} N | l {x.min():.2f}-{x.max():.2f}, loaded median {np.median(x[w]):.2f}, "
          f"{100 * np.mean(x[w] > 1):3.0f}% of loaded samples past l_opt | v {y.min():+.2f}..{y.max():+.2f}, "
          f"loaded {y[w].min():+.2f}..{y[w].max():+.2f}")
pen = mus["pennation"]
for m in ("soleus_r", "gasmed_r", "vaslat_r"):
    i = names.index(m)
    j = np.nanargmax(np.where(ok, F[:, i], -1))
    print(f"{m}: pennation at its peak force {np.degrees(pen[j, i]):.1f} deg -> 1/cos = {1 / np.cos(pen[j, i]):.3f}")

soleus_r    peak  2941 N | l 0.52-1.09, loaded median 1.02,  66% of loaded samples past l_opt | v -0.64..+0.38, loaded -0.48..+0.18
vaslat_r    peak  2291 N | l 0.72-1.09, loaded median 0.88,   0% of loaded samples past l_opt | v -0.30..+0.25, loaded -0.11..+0.18
gasmed_r    peak  2200 N | l 0.48-1.15, loaded median 1.09,  97% of loaded samples past l_opt | v -0.78..+0.34, loaded -0.40..+0.07
psoas_r     peak  1390 N | l 0.98-1.19, loaded median 1.16,  95% of loaded samples past l_opt | v -0.08..+0.07, loaded -0.08..+0.05
recfem_r    peak  1305 N | l 0.47-1.42, loaded median 1.00,  50% of loaded samples past l_opt | v -0.52..+0.37, loaded -0.10..+0.32
tibant_r    peak  1046 N | l 0.92-1.30, loaded median 1.12,  94% of loaded samples past l_opt | v -0.25..+0.42, loaded -0.25..+0.31
glmax2_r    peak   946 N | l 0.65-0.99, loaded median 0.95,   0% of loaded samples past l_opt | v -0.12..+0.16, loaded -0.12..-0.01
semimem_r   peak   940 N | l 0.56-1.39, loaded median 1.32,  90% of loaded s

What I take from this:

- **Most of the big muscles work a bit past $l_{opt}$ when they're loaded**, on the descending
  limb. Gastrocnemius medialis (median 1.09 while loaded, 97% of those samples past $l_{opt}$),
  psoas (1.16, 95%), iliacus (1.16, 100%), tibialis anterior (1.12, 94%), and the hamstrings are
  the extreme: semimembranosus 1.32 (90%) and biceps femoris long head 1.29 (96%). With the
  published width, $f_L(1.32)$ would be 0.54. With our 1.5 width it's 0.77. So the widening matters
  most for exactly these muscles. (If passive force were switched on, the hamstrings at 1.3-1.4
  would also start pulling passively. I ignored that, so take their forces with salt.)
- **The vasti and gluteus maximus work on the ascending side**: vastus lateralis median 0.88
  (never past $l_{opt}$ while loaded), glmax2 0.95, piriformis 0.79 (and down to 0.23 at its
  shortest, which is off the useful part of the curve).
- **Velocities are slow.** Everything stays within $-0.78$ to $+0.45$ of $v_{max}$ over the whole
  window, and while loaded mostly within about $\pm0.3$. The fastest shortening is in the
  triceps surae during push-off and swing (gasmed down to $-0.78$, gaslat $-0.67$, soleus
  $-0.64$), but while it's loaded the soleus only reaches $-0.48$ (where $f_V = 0.25$) and gasmed
  $-0.40$. With $v_{max} = 10\, l_{opt}$/s, walking just doesn't ask much of $f_V$. That's also part
  of why static optimization (which knows $f_V$ but not activation dynamics) can keep up.

### Static vs dynamic optimization, stated precisely

**Static optimization** (OpenSim `AnalyzeTool` + `StaticOptimization`). Each of the 613 frames
(0.10&ndash;5.20 s) is its own little problem, with no memory of the frame before:

$$\min_{a,\,u}\; \sum_m a_m^2 + \sum_r u_r^2 \quad \text{s.t.}\quad \ddot q_{model}(q, \dot q;\; a_m F_{iso,m} f_L f_V,\; u_r F^{opt}_r,\; \text{GRF}) = \ddot q_{IK}, \qquad 0.01 \le a_m \le 1$$

The constraint (accelerations of the model equal the IK accelerations) is the same as "muscle
moments + reserve moments = the inverse-dynamics moment" for every coordinate. $q, \dot q, \ddot q$
come from the 12 Hz-filtered IK at that instant, and `use_muscle_physiology = true` means $f_L$ and
$f_V$ are evaluated at those fibre lengths and velocities, so a muscle that's short or shortening
fast is "weaker" per unit activation. There's no activation dynamics: $a$ can jump from 0.01 to 1
between two frames 8.3 ms apart. Activation exponent 2, convergence 1e-4, 58 s for the whole trial.
One detail I only found by rebuilding the numbers: SO's reported force is
$a\,F_{iso}\,f_L f_V$ *with no $\cos\alpha$ and no damping* (reconstructs to $4\times10^{-5}$ N), and
that's what it applies along the tendon (its muscle moment matches ID to 0.006/0.06/0.01 N m RMS at
ankle/knee/hip). That makes a pennate muscle about $1/\cos\alpha$ stronger in SO than in Moco:
7.6% for soleus at its peak (21.6&deg;), 4% for vastus lateralis, 1% for gasmed.

**Dynamic optimization** (OpenSim Moco `MocoInverse`). One problem over the whole 5.1 s window:

$$\min_{e,\,a,\,u}\; \int_{0.10}^{5.20} \Big(\sum_m e_m^2 + \sum_r u_r^2 + \sum_m a_m^2\Big)\,dt$$

subject to: the kinematics prescribed from the same filtered IK (`PositionMotion`), the
**implicit multibody dynamics** $M(q)\ddot q + C(q,\dot q) - G(q) = \tau_{muscles}(a, q, \dot q) + \tau_{reserves}(u) + \tau_{GRF}$
at every collocation point, the **activation ODE** above linking each $e_m$ to $a_m$, and
$0.01 \le e_m, a_m \le 1$. The states are just the 80 muscle activations (both legs); the controls are
the 80 excitations plus the reserve/residual signals. It's solved by direct collocation:
Hermite&ndash;Simpson on 256 mesh intervals of 20 ms, CasADi + IPOPT with a limited-memory Hessian,
convergence and constraint tolerances 1e-3. It took 335 iterations and 1087 s (about 19 times SO),
objective 153.0, `Solve_Succeeded`. The ODE and the dynamics are constraints on the collocation
points, not things it integrates forward. Reserves on the leg joints have optimal force 1 N m,
and the pelvis residuals 100, same as in SO.

**Why they come out close.** Same model, same IK, same GRF, same reserves, and the kinematics
are prescribed in both, so the fibre lengths and velocities are identical: on the Moco mesh,
SO's $\tilde l^M$ and MocoInverse's differ by 2.4e-4 RMS (99th percentile 6.3e-4) and $\tilde v^M$ by
7.3e-4 RMS (99th percentile 3.5e-3). The worst spots are single samples of glmax1 (0.019 in length at
0.43 s, 0.009 in velocity at 4.26 s), which I think comes from the two tools filtering and splining
the same IK file slightly differently (AnalyzeTool's own low-pass vs Moco's `TabOpLowPassFilter`),
around glmax1's wrap surface. The tendon length is $l_s$ in both (difference 5e-9 m). The
objective is nearly the same too (squared activations in both; Moco adds squared excitations).
So the only real new ingredient is activation dynamics, and with $\tau$ of 10/40 ms against a
gait cycle of about a second it doesn't bite much (Anderson & Pandy 2001, *J Biomech* 34:153,
"Static and dynamic optimization solutions for gait are practically equivalent", 815 citations).

**But "close" is about the joint moments, not every muscle.** Both muscle sets add up to the ID
moment (SO to 0.06 N m RMS or better, MocoInverse to 1.4/1.4/1.8 N m at ankle/knee/hip, with the
reserves taking the rest), and the per-muscle time courses correlate at a median of 0.89 (from 0.53
for addmagIsch to 0.99). Synergists can still trade load: soleus peaks at 1984 N in SO and 2941 N
in Moco, and at Moco's soleus peak (1.92 s) Moco has gasmed at 741 N where SO has it at 1827 N.
Same ankle moment, split differently inside the triceps surae. Some of that is activation dynamics (a muscle that
can't switch off in time carries load earlier and later), some is the $\cos\alpha$ and damping
terms that only Moco's force contains, and some is just that the problem has a flat bottom.

**What would pull them apart**:

- a **compliant tendon**: then fibre length is a state, $l^M \ne$ geometry, and the
  ankle plantarflexors in particular would work at very different $\tilde l^M, \tilde v^M$ (the
  Achilles stretches several percent). SO can't do that properly at all.
- a **different objective**: e.g. minimizing metabolic energy, or $a^3$, or adding a
  smoothness term, would redistribute load among synergists in both, but differently.
- **tracking or predictive (forward-dynamics) problems** (`MocoTrack`, predictive simulation):
  then the kinematics aren't prescribed, the muscles have to *produce* the motion, and the
  time history really matters.
- **slower activation** (bigger $\tau$, or $b = 10$ so deactivation is really 40 ms-ish): more
  co-contraction around fast torque reversals, which SO can't see.
- **fast movements** like running or jumping, where $|\tilde v^M|$ gets big enough that $f_V$ and
  activation timing both start to matter.

## Knee contact forces: how hard the femur pushes on the right tibia, total and medial vs lateral

Muscle forces are only half the story for the knee. What the cartilage actually feels is
the contact force, and that's the muscles plus the ground reaction plus the leg's own
inertia, all squeezed through the joint. So here I take both muscle solutions (static and
dynamic optimization) and compute the right tibiofemoral contact force.

What I assumed (plainly):

- **The reaction is a Newton&ndash;Euler balance of everything below the knee** (tibia + foot):
  IK accelerations, gravity, the measured ground force and free moment, and every muscle
  path segment that pulls on the shank or foot. I didn't use OpenSim's JointReaction
  analysis, because it runs forward dynamics (its accelerations are whatever the forces
  make). Without muscles, the same code reproduces OpenSim's ID knee moment
  (r = &minus;0.999, the minus is just axis direction).
- **It's expressed in the tibia frame**: compressive force along the tibia's long axis,
  AP shear, and the frontal-plane (varus/valgus) moment, about the knee centre.
- **Two point contacts on a flat plateau**, one per condyle, &plusmn;d<sub>ic</sub>/2 either
  side of the knee centre, perpendicular to the tibial axis (Winby et al. 2009 J Biomech,
  387 citations; Lerner et al. 2015 J Biomech, 228 citations, generic &plusmn;20 mm).
  d<sub>ic</sub> = 40 mm &times; P1's tibia scale factor = **40.3 mm**. Then
  F<sub>med</sub> + F<sub>lat</sub> = F<sub>compressive</sub> and
  (F<sub>med</sub> &minus; F<sub>lat</sub>)&middot;d<sub>ic</sub>/2 = varus moment.
- **Ligaments carry nothing.** If a condyle would need to pull, I call it lift-off,
  clamp it to zero and report the ligament tension it would take, separately.
- The Edwards lab (Calgary) does the same kind of thing for the tibia: muscle forces plus
  the ID reaction in the tibia frame (Baggaley et al. 2024 J Exp Biol, 11 citations;
  Baggaley et al. 2022 J Biomech Eng, 20 citations). Their static optimization minimizes
  squared muscle *stress* instead of squared activation; I kept the existing solves. They
  don't split medial/lateral, so the split here follows Winby and Lerner.

Body weight is P1's 85.16 kg, so 1 BW = 835 N.

In [8]:
import json
import warnings
import matplotlib.pyplot as plt

KC = np.load("../data/opensim/p1_trial13_knee_contact.npz")
KJ = json.load(open("../data/opensim/p1_trial13_knee_contact.json"))
st = KC["stance_frames"]
s = np.linspace(0, 100, 101)

def stance_curves(x):
    # [n_stances, 101], each right stance resampled to 0-100 %
    return np.array([np.interp(np.linspace(a, b, 101), np.arange(a, b + 1), x[a:b + 1]) for a, b in st])

def show(fig):
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

base = KJ["baseline"]
for k, lab in (("so", "static opt"), ("dyn", "dynamic opt")):
    b = base[k]
    print(f"{lab:12s} peak total {b['peak_total_BW']['mean_of_stance_peaks']:.2f} BW, medial "
          f"{b['peak_medial_BW']['mean_of_stance_peaks']:.2f}, lateral {b['peak_lateral_BW']['mean_of_stance_peaks']:.2f} "
          f"(mean of {b['peak_total_BW']['n_stances']} stance peaks); medial share at the total peak "
          f"{100 * b['medial_share_at_total_peak']:.0f}%, lateral lift-off frames in stance: "
          f"{b['lateral_liftoff_frames_in_stance']} of {b['stance_frames']}")
print(f"d_ic = {1000 * float(KC['d_ic']):.1f} mm, body mass {float(KC['body_mass']):.2f} kg")

static opt   peak total 5.83 BW, medial 4.00, lateral 2.08 (mean of 5 stance peaks); medial share at the total peak 67%, lateral lift-off frames in stance: 0 of 348
dynamic opt  peak total 6.13 BW, medial 3.98, lateral 2.57 (mean of 5 stance peaks); medial share at the total peak 62%, lateral lift-off frames in stance: 0 of 348
d_ic = 40.3 mm, body mass 85.16 kg


In [9]:
SOC, DYC = R.JOINT_COLORS[0], R.JOINT_COLORS[1]
fig, axes = R.new_grid(1, 4, height=3.1, width=12.5)
ax = axes[0]
for j, (key, title) in enumerate((("total", "total (compressive)"), ("medial", "medial condyle"),
                                  ("lateral", "lateral condyle"))):
    ax = axes[0, j]
    for m, c, lab in (("so", SOC, "static opt"), ("dyn", DYC, "dynamic opt")):
        cv = stance_curves(KC[f"{m}_contact_{key}"])
        ax.fill_between(s, cv.mean(0) - cv.std(0), cv.mean(0) + cv.std(0), color=c, alpha=0.15, lw=0)
        ax.plot(s, cv.mean(0), color=c, lw=1.8, label=lab)
    R.style_axes(ax, "right stance (%)", "contact force (BW)" if j == 0 else "")
    ax.set_title(title, fontsize=10, color=R.NEUTRAL)
ref = KJ["in_vivo_reference"]
axes[0, 0].axhline(ref["total_walking_peak_BW"], color=R.NEUTRAL, ls="--", lw=1)
axes[0, 0].text(2, ref["total_walking_peak_BW"] + 0.12, "in vivo walking peak 2.61 BW (Kutzner 2010)", fontsize=7.5, color=R.NEUTRAL)
axes[0, 1].axhspan(*ref["medial_peaks_BW"], color=R.FAINT, alpha=0.6, lw=0)
axes[0, 1].text(2, ref["medial_peaks_BW"][1] + 0.12, "in vivo medial peaks 1.76-1.87 BW (Kutzner 2013)", fontsize=7.5, color=R.NEUTRAL)
ymax = max(ax.get_ylim()[1] for ax in axes[0, :3])
for ax in axes[0, :3]:
    ax.set_ylim(-0.1, ymax)
axes[0, 0].legend(frameon=False, fontsize=8, loc="upper right")
ax = axes[0, 3]
for m, c in (("so", SOC), ("dyn", DYC)):
    cv = stance_curves(100 * KC[f"{m}_medial_share"])
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")           # share is undefined while the knee is unloaded
        ax.plot(s, np.nanmean(cv, 0), color=c, lw=1.8)
ax.axhspan(65, 73, color=R.FAINT, alpha=0.6, lw=0)
ax.text(2, 75, "in vivo 65-73% at the peaks (Halder 2012)", fontsize=7.5, color=R.NEUTRAL)
ax.set_ylim(0, 100)
R.style_axes(ax, "right stance (%)", "medial share (%)")
ax.set_title("medial / total", fontsize=10, color=R.NEUTRAL)
show(fig)

How this compares with in vivo knees. The shapes are right (two humps, medial carries most of
it), but the magnitudes are big: peak total **5.83 BW** (static) and **6.13 BW**
(dynamic), vs **2.61 BW** measured in instrumented knees during walking (Kutzner et al. 2010
J Biomech, 817 citations), and vs 1.8&ndash;3.0 BW for most modelling papers (Fregly et al.
2012 J Orthop Res, 622 citations, who also note models tend to over-predict). Medial peaks
are **4.00 / 3.98 BW** vs 1.76&ndash;1.87 BW in vivo (Kutzner et al. 2013 PLoS One,
253 citations). The medial share at the total peak is **67% / 62%**, which is close to
the in vivo 65&ndash;73% (Halder et al. 2012 JBJS, 155 citations; static is inside it, dynamic a bit under).

So the split looks believable and the total is about 2.2&ndash;2.3&times; too high. I think most
of that is real-subject vs implant-patient: P1's peak knee extension moment here is ~134 N m
(OpenSim ID, 1.58 N m/kg), a lot more than elderly TKA patients walk with, and the
quadriceps has to supply all of it with a ~4-5 cm moment arm. Static optimization also
doesn't add any co-contraction, so if anything it should come out *low*; the excess has to come
from the high knee moment, the rigid-tendon muscles and the model's moment arms, not from
over-recruiting. Dynamic optimization adds a little (6.13 vs 5.83 BW), mostly
on the lateral side.

## Pushing the knee outward: a varus sweep, to see how much load moves to the medial side

This is a made-up deformity. I take P1's walking and push the right knee **laterally by
d = 0, 5, 10, &hellip; 50 mm**, perpendicular to the hip&ndash;ankle line (same geometry as the
AFF report's `varus_shift`: hip, ankle, foot and the measured ground force stay put).

What I did, exactly:

- **Markers**: the thigh markers (RGTR, RTH1-3, and the epicondyles RLEP/RMEP) are rotated
  rigidly about the hip centre, and the shank markers (RSH1-3, malleoli RLML/RMML) about the
  ankle centre, so the mid-epicondyle point moves out by d. That's every marker that should
  ride with the knee; the foot markers don't move.
- **Model**: the Rajagopal knee only flexes, so a marker shift by itself would just turn into
  IK error. I gave the model a matching fixed varus: the knee joint's tibia-side frame is
  rotated about the tibia's AP axis, through the knee centre, by
  &theta; = d(1/L<sub>thigh</sub> + 1/L<sub>shank</sub>) with L = 0.412 / 0.394 m from the data.
  That's 1.42&deg; per 5 mm, so **14.2&deg; at 50 mm** (which is a lot, honestly).
- **Check**: after IK, the model's knee centre moved laterally off its hip&ndash;ankle line by
  5.1 / 25.2 / 50.4 mm (mean over the trial) for d = 5, 25, 50 mm, and the IK marker RMS stayed at
  14.2&ndash;14.5 mm. So the model really did take the deformity.
- Then ID, static optimization and the same contact split, for every d. Dynamic optimization
  (MocoInverse, ~25 min each) only at 0, 25, 50 mm.

One naming thing: varus/valgus (adduction) moments live in the **frontal** plane, not the
sagittal one. Below I show both: the frontal moment that does the medial/lateral shifting,
and the sagittal (flexion/extension) moment, which barely moves.

In [10]:
d_mm = 1000 * KC["sweep_d_m"]
th = KC["sweep_theta_deg"]
tab = KJ["sweep"]["table"]
print(" d(mm) varus(deg) | SO peak total medial lateral (BW) | lateral lift-off frames, ligament (BW) | KAM OpenSim/boneid (N m) | knee ext moment OpenSim/boneid (N m)")
for r in tab:
    print(f"{r['d_mm']:5d} {r['theta_deg']:9.2f}  | {r['peak_so_total_BW']:6.2f} {r['peak_so_medial_BW']:6.2f} {r['peak_so_lateral_BW']:6.2f}"
          f"            | {r['lateral_liftoff_stance_frames']:4d}, {r['peak_lateral_ligament_tension_BW']:.2f}"
          f"                    | {r['peak_kam_opensim_Nm']:6.1f} / {r['peak_kam_boneid_Nm']:6.1f}"
          f"          | {r['peak_knee_extension_moment_opensim_Nm']:6.1f} / {r['peak_knee_extension_moment_boneid_Nm']:6.1f}")
for v in KJ["sweep"]["ik_verification"]:
    if v["d_mm"] in (5, 25, 50):
        print(f"IK check d={v['d_mm']} mm: model knee moved {v['model_knee_lateral_shift_mean_mm']:.2f} "
              f"+/- {v['model_knee_lateral_shift_sd_mm']:.2f} mm, IK RMS {v['ik_rms_mean_mm']:.2f} mm")

 d(mm) varus(deg) | SO peak total medial lateral (BW) | lateral lift-off frames, ligament (BW) | KAM OpenSim/boneid (N m) | knee ext moment OpenSim/boneid (N m)
    0      0.00  |   5.83   4.00   2.08            |    0, 0.00                    |   68.9 /   76.1          |  134.4 /  111.9
    5      1.42  |   5.85   4.18   1.95            |    0, 0.00                    |   74.5 /   82.0          |  136.3 /  112.2
   10      2.85  |   5.88   4.38   1.81            |    0, 0.00                    |   80.2 /   87.8          |  138.1 /  112.5
   15      4.27  |   5.90   4.57   1.68            |    0, 0.00                    |   85.8 /   93.7          |  139.7 /  112.8
   20      5.69  |   5.91   4.75   1.54            |    4, 0.06                    |   91.4 /   99.7          |  141.2 /  113.1
   25      7.12  |   5.93   4.94   1.40            |    7, 0.13                    |   97.0 /  105.9          |  142.6 /  113.5
   30      8.54  |   5.94   5.13   1.27            |   19, 0.20        

In [11]:
def peaks(arr):
    return np.array([np.mean([np.nanmax(row[a:b + 1]) for a, b in st]) for row in arr])

fig, axes = R.new_grid(1, 2, height=3.4, width=12.5)
ax = axes[0, 0]
for key, c, lab in (("total", R.NEUTRAL, "total"), ("medial", R.JOINT_COLORS[0], "medial"),
                    ("lateral", R.JOINT_COLORS[1], "lateral")):
    ax.plot(d_mm, peaks(KC[f"sweep_so_{key}"]), "o-", color=c, lw=1.8, ms=4, label=f"{lab}, static opt")
    if len(KC["sweep_dyn_d_m"]):
        ax.plot(1000 * KC["sweep_dyn_d_m"], peaks(KC[f"sweep_dyn_{key}"]), "s", mfc="none", color=c, ms=7,
                label=f"{lab}, dynamic opt")
ax.plot(d_mm, peaks(KC["sweep_so_lig_lateral"]), "^:", color=R.JOINT_COLORS[3], lw=1.2, ms=4,
        label="lateral ligament tension needed")
R.style_axes(ax, "knee pushed laterally, d (mm)", "peak in stance (BW)")
ax.legend(frameon=False, fontsize=7.5, ncol=2, loc="center right")
ax.set_ylim(bottom=0)
sec = ax.secondary_xaxis("top", functions=(lambda x: x * th[-1] / d_mm[-1], lambda v: v * d_mm[-1] / th[-1]))
sec.set_xlabel("added varus (deg)", color=R.NEUTRAL, fontsize=9)
sec.tick_params(colors=R.NEUTRAL, labelsize=8)
ax.set_title("contact force peaks", fontsize=10, color=R.NEUTRAL)

ax = axes[0, 1]
ax.plot(d_mm, peaks(KC["sweep_frontal_moment_os"]), "o-", color=R.JOINT_COLORS[0], lw=1.8, ms=4,
        label="frontal: external adduction (varus), OpenSim")
ax.plot(d_mm, peaks(KC["sweep_kam_boneid"]), "o--", color=R.JOINT_COLORS[0], mfc="none", lw=1.2, ms=4,
        label="frontal: external adduction, boneid varus_shift")
ax.plot(d_mm, peaks(-KC["sweep_knee_sagittal_os"]), "o-", color=R.JOINT_COLORS[2], lw=1.8, ms=4,
        label="sagittal: knee extension, OpenSim ID")
ax.plot(d_mm, peaks(-KC["sweep_knee_sagittal_boneid"]), "o--", color=R.JOINT_COLORS[2], mfc="none", lw=1.2, ms=4,
        label="sagittal: knee extension, boneid")
R.style_axes(ax, "knee pushed laterally, d (mm)", "peak in stance (N m)")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
sec = ax.secondary_xaxis("top", functions=(lambda x: x * th[-1] / d_mm[-1], lambda v: v * d_mm[-1] / th[-1]))
sec.set_xlabel("added varus (deg)", color=R.NEUTRAL, fontsize=9)
sec.tick_params(colors=R.NEUTRAL, labelsize=8)
ax.set_title("knee moments (frontal vs sagittal)", fontsize=10, color=R.NEUTRAL)
show(fig)

In [12]:
SHOW = [0, 25, 50]
cols = {0: R.NEUTRAL, 25: R.JOINT_COLORS[0], 50: R.JOINT_COLORS[1]}
fig, axes = R.new_grid(1, 4, height=3.1, width=12.5)
for d in SHOW:
    i = list(d_mm.astype(int)).index(d)
    lab = f"d = {d} mm ({th[i]:.1f} deg)"
    for j, (key, title) in enumerate((("sweep_so_medial", "medial contact (static opt)"),
                                      ("sweep_so_lateral", "lateral contact (static opt)"))):
        axes[0, j].plot(s, stance_curves(KC[key][i]).mean(0), color=cols[d], lw=1.8, label=lab)
        axes[0, j].set_title(title, fontsize=10, color=R.NEUTRAL)
    axes[0, 1].plot(s, stance_curves(KC["sweep_so_lig_lateral"][i]).mean(0), color=cols[d], lw=1, ls=":")
    axes[0, 2].plot(s, stance_curves(KC["sweep_frontal_moment_os"][i]).mean(0), color=cols[d], lw=1.8)
    axes[0, 3].plot(s, stance_curves(-KC["sweep_knee_sagittal_os"][i]).mean(0), color=cols[d], lw=1.8)
ymax = max(axes[0, j].get_ylim()[1] for j in (0, 1))
for j in (0, 1):
    axes[0, j].set_ylim(-0.1, ymax)
    R.style_axes(axes[0, j], "right stance (%)", "BW" if j == 0 else "")
axes[0, 1].text(40, ymax * 0.9, "dotted: lateral ligament\ntension needed (lift-off)", fontsize=7.5, color=R.NEUTRAL)
axes[0, 0].legend(frameon=False, fontsize=7.5)
axes[0, 2].set_title("external adduction moment (frontal)", fontsize=10, color=R.NEUTRAL)
axes[0, 3].set_title("knee extension moment (sagittal)", fontsize=10, color=R.NEUTRAL)
lo = min(axes[0, j].get_ylim()[0] for j in (2, 3)); hi = max(axes[0, j].get_ylim()[1] for j in (2, 3))
for j in (2, 3):
    axes[0, j].set_ylim(lo, hi)
    R.style_axes(axes[0, j], "right stance (%)", "N m" if j == 2 else "")
show(fig)

What actually happens, per 10 mm of lateral knee shift (about 2.85&deg; of varus):

- **Frontal moment goes up linearly**: the external adduction moment peak rises from
  68.9 to 124.7 N m over 0&ndash;50 mm (OpenSim), about 1.12 N m per mm, which is
  roughly the vertical force (~1 kN at the peak) times d, as expected. boneid's own
  `varus_shift` gives the same slope (76.1 &rarr; 137.4 N m; its baseline is a bit
  higher because its knee centre and thigh frame aren't the OpenSim model's).
- **Medial goes up, lateral goes down**: medial peak 4.00 &rarr; 5.90 BW, lateral
  2.08 &rarr; 0.86 BW (static opt). That's ~112 N more medial and ~72 N less
  lateral per degree, about double Lerner's 51 / 30 N per degree, which fits our loads being
  about double too.
- **Total barely changes**: 5.83 &rarr; 5.97 BW. The muscles don't find a cheaper way
  to hold the varus knee; the load just slides across.
- **The lateral condyle starts lifting off at d = 20 mm** (5.7&deg;): a few stance frames
  where the lateral side would have to pull. By 50 mm it's 231 of 348 stance frames, and
  a lateral ligament would need up to 0.69 BW. With ligaments carrying nothing, that's the
  model saying "this knee needs its LCL".
- **Sagittal moment hardly moves**: peak knee extension moment 134 &rarr; 148 N m
  (OpenSim, +10%), and 112 &rarr; 116 N m in boneid. So the sweep is almost
  purely a frontal-plane effect, like I'd expect.
- **Dynamic optimization agrees on the trend** (d = 0 mm: total 6.13, medial 3.98, lateral 2.57 BW; d = 25 mm: total 6.21, medial 4.89, lateral 1.83 BW; d = 50 mm: total 6.22, medial 5.89, lateral 1.29 BW). Its lateral side stays a bit higher than static optimization's, same as at baseline.